# Sample Practical Exam: Household Energy Usage Prediction

EnergyTech Solutions is a rapidly expanding global energy management company operating in over 20 countries across Europe, North America, and Asia. They specialize in smart home energy systems, renewable energy integrations, and energy-efficient appliances, combining direct sales with a marketplace for third-party providers.

The company has experienced significant growth due to its data-driven approach to energy optimization. As they plan expansion into new markets, they aim to enhance their ability to predict household energy consumption.

Their analytics team seeks to predict which households are likely to have high energy usage based on their energy consumption patterns. As an AI Engineer, you will develop this prediction system. Your work will directly support EnergyTech’s growth strategy and their mission to promote sustainable energy use.

### Data Description

| Column Name        | Criteria                                                                 |
|--------------------|--------------------------------------------------------------------------|
| household_id       | Integer. Unique identifier for each household. No missing values.        |
| time_of_use        | Float. Hours of appliance use per day. Missing values replaced with median. |
| appliances_used    | Integer. Number of appliances used daily. Missing values replaced with mode. |
| energy_cost        | Float. Daily energy cost in dollars. Missing values replaced with 0.      |
| home_type          | String. One of: Apartment, House, Condo. Missing values replaced with "Other". |
| occupancy_type     | String. One of: Single, Family. Missing values replaced with "Single".    |
| energy_usage       | Binary. High energy usage (1) or low (0). Target variable.               |


## Task 1

The analytics team has collected household energy data in `raw_energy_data.csv`, but it contains missing values and inconsistencies. Create a cleaned version of the dataframe:

- Start with the data in `raw_energy_data.csv`.
- Your output should be a DataFrame named `clean_data`.
- All column names and values should match the table above.

| Column Name        | Criteria                                                                 |
|--------------------|--------------------------------------------------------------------------|
| household_id       | Integer. Unique identifier for each household. No missing values.        |
| time_of_use        | Float. Hours of appliance use per day. Missing values replaced with median. |
| appliances_used    | Integer. Number of appliances used daily. Missing values replaced with mode. |
| energy_cost        | Float. Daily energy cost in dollars. Missing values replaced with 0.      |
| home_type          | String. One of: Apartment, House, Condo. Missing values replaced with "Other". |
| occupancy_type     | String. One of: Single, Family. Missing values replaced with "Single".    |
| energy_usage       | Binary. High energy usage (1) or low (0). Target variable.               |


In [5]:
# Write your answer to Task 1 here 
import pandas as pd

clean_data = pd.read_csv("raw_energy_data.csv")

# household_id: integer, no missing
clean_data = clean_data.dropna(subset=["household_id"])
clean_data["household_id"] = clean_data["household_id"].astype(int)

# time_of_use: float, missing -> median
clean_data["time_of_use"] = pd.to_numeric(clean_data["time_of_use"], errors="coerce")
clean_data["time_of_use"] = clean_data["time_of_use"].fillna(clean_data["time_of_use"].median()).astype(float)

# appliances_used: integer, missing -> mode
clean_data["appliances_used"] = pd.to_numeric(clean_data["appliances_used"], errors="coerce")
clean_data["appliances_used"] = clean_data["appliances_used"].fillna(clean_data["appliances_used"].mode()[0]).astype(int)

# energy_cost: float, missing -> 0
clean_data["energy_cost"] = pd.to_numeric(clean_data["energy_cost"], errors="coerce").fillna(0).astype(float)

# home_type: Apartment, House, Condo, else "Other"
clean_data["home_type"] = clean_data["home_type"].astype(str).str.strip().str.title()
clean_data.loc[~clean_data["home_type"].isin(["Apartment", "House", "Condo"]), "home_type"] = "Other"

# occupancy_type: Single, Family, else "Single"
clean_data["occupancy_type"] = clean_data["occupancy_type"].astype(str).str.strip().str.title()
clean_data.loc[~clean_data["occupancy_type"].isin(["Single", "Family"]), "occupancy_type"] = "Single"

# energy_usage: binary 0/1
clean_data["energy_usage"] = clean_data["energy_usage"].astype(int)

clean_data

,household_id,time_of_use,appliances_used,energy_cost,home_type,occupancy_type,energy_usage
0,1,9.610,2,3.94,House,Single,0
1,2,22.870,6,6.14,House,Single,1
2,3,17.840,12,0.00,Apartment,Family,1
3,4,14.770,5,6.54,House,Family,1
4,5,4.590,13,9.18,Other,Family,1
...,...,...,...,...,...,...,...
1495,1496,16.370,7,5.02,Apartment,Single,0
1496,1497,15.250,2,4.37,Other,Single,0
1497,1498,12.675,13,2.03,Condo,Family,1
1498,1499,9.740,9,6.74,House,Single,1


## Task 2

With all preparatory work complete, create and train a neural network to predict energy usage.

- Using PyTorch, create a network with:
  - Two hidden layers: the first with 16 units, the second with 8 units.
  - ReLU activation for the hidden layers.
  - Sigmoid activation for the output layer.
- Train the model using prepared features in `input_model_features.csv` to predict `energy_usage`.
- Use the validation dataset `validation_features.csv` to predict new values.
- Your model should be named `energy_model`, and your output should be a DataFrame named `validation_predictions` with columns `household_id` and `energy_usage`. The `energy_usage` column must contain your predicted values.

In [6]:
# Write your answer to Task 2 here
import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(42)

train = pd.read_csv("input_model_features.csv")
val = pd.read_csv("validation_features.csv")

feature_cols = [c for c in train.columns if c not in ["household_id", "energy_usage"]]

X = torch.tensor(train[feature_cols].values, dtype=torch.float32)
y = torch.tensor(train["energy_usage"].values, dtype=torch.float32).view(-1, 1)
X_val = torch.tensor(val[feature_cols].values, dtype=torch.float32)

energy_model = nn.Sequential(
    nn.Linear(len(feature_cols), 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(energy_model.parameters(), lr=0.01)

for epoch in range(500):
    optimizer.zero_grad()
    loss = criterion(energy_model(X), y)
    loss.backward()
    optimizer.step()

energy_model.eval()
with torch.no_grad():
    preds = (energy_model(X_val) >= 0.5).int().view(-1).numpy()

validation_predictions = pd.DataFrame({
    "household_id": val["household_id"],
    "energy_usage": preds
})

validation_predictions

,household_id,energy_usage
0,1501,0
1,1502,1
2,1503,0
3,1504,1
4,1505,0
...,...,...
495,1996,0
496,1997,0
497,1998,0
498,1999,0
